# 01 — Flow dataset EDA

This notebook inspects the downloaded, labelled flow dataset before modelling. The stratified sample is deterministic. Identifiers are retained here for graph construction and talker analysis, but are excluded from numeric model features in `src/preprocess.py`.

In [ ]:
from pathlib import Path
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from src.data import load_flows, find_column

cfg = yaml.safe_load(Path('configs/experiment.yaml').read_text())
flows = load_flows(cfg['data']['path'], sample=cfg['data'].get('sample'), seed=cfg['data'].get('sample_seed', 17))
print(f"{len(flows):,} sampled rows; time source: {flows['_time_source'].iloc[0]}")
display(flows.head())

## Binary balance and attack categories

Because attack prevalence is low, report both counts and proportions. The project samples proportionally by the binary label so the CPU subset does not artificially balance away the operational class imbalance.

In [ ]:
counts = flows['_attack_type'].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
flows['_target'].value_counts().sort_index().rename(index={0:'Benign', 1:'Attack'}).plot.bar(ax=axes[0], color=['#277da1','#f94144'])
axes[0].set(title='Binary class counts', ylabel='Flows')
counts.head(12).sort_values().plot.barh(ax=axes[1], color='#577590')
axes[1].set(title='Most frequent attack labels', xlabel='Flows')
plt.tight_layout()

## Top talkers

The source and destination addresses are used only to construct host-level graph structure. They are not passed as raw numeric model features.

In [ ]:
src_col = find_column(flows, ('Source IP','srcip','src_ip','IPV4_SRC_ADDR'), required=False)
dst_col = find_column(flows, ('Destination IP','dstip','dst_ip','IPV4_DST_ADDR'), required=False)
if src_col and dst_col:
    display(flows[src_col].value_counts().head(10).rename('outgoing_flows').to_frame())
    display(flows[dst_col].value_counts().head(10).rename('incoming_flows').to_frame())
else:
    print('This dataset file has no endpoint identifiers; graph modelling must use a different source file.')

## Feature distributions

Flow counters and durations are typically heavy-tailed. The engineered pipeline applies `log1p` to non-negative count-like features before standardization; all fitted transforms are learned only on the training period.

In [ ]:
numeric_cols = flows.select_dtypes(include='number').columns.tolist()
selected = [c for c in numeric_cols if any(k in c.lower() for k in ('byte','packet','duration','pkts','sbytes','dbytes'))][:4]
if selected:
    flows[selected].replace([float('inf'), -float('inf')], np.nan).hist(bins=40, figsize=(12, 7), log=True)
    plt.suptitle('Raw heavy-tailed flow features (log count axis)')
    plt.tight_layout()

## Temporal coverage and split

The main scripts parse CIC timestamps as day/month/year UTC and group the sorted flows into whole one-hour windows. These windows are assigned chronologically 60/20/20 to train/validation/test. Validation selects the autoencoder threshold; only the final period is used for reported test metrics.

In [ ]:
if '_order' in flows:
    print('Ordered range:', pd.to_datetime([flows['_order'].min(), flows['_order'].max()], unit='ns', utc=True))
    display(pd.crosstab(flows['_time_source'], flows['_target'], normalize='columns'))
else:
    print('No usable timestamp was found; any order-based split would be a proxy only.')